In [ ]:

import sys
import glob
import numpy as np
import os
from rdkit import Chem
import pandas as pd
from rdkit.Chem.PandasTools import LoadSDF
def initAlkene(smiles , addHs):
    molec = Chem.MolFromSmiles(smiles)
    double = 0
    for bond in molec.GetBonds():
        atom1 = bond.GetBeginAtom()
        atom2 = bond.GetEndAtom()
        if atom1.GetAtomicNum() == 6 and atom2.GetAtomicNum() == 6 and bond.GetBondType() == Chem.BondType.DOUBLE:
            double +=1
            c1 = bond.GetBeginAtomIdx()
            c2 = bond.GetEndAtomIdx()
            CC = [c1 , c2]
    if double != 1:
        raise SystemError("More than 1 Alkene found in system")
    elif addHs:
        mol_with_hs = Chem.AddHs(molec)
        return CC , mol_with_hs
    else:
        return CC , molec 
def breadthFirstSearch(molec, period, contactHash, forbiddenAtoms):
    # contactHash: { 0: [[startAtom]], 1: [[startAtom, id1], [startAtom, id2]], 2: [[startAtom, id1, id4], ...] }
    currentPeriod = list(contactHash.keys())[-1]

    if currentPeriod > period:
        return contactHash
    else:
        prevTraversed = set(atom for paths in contactHash.values() for path in paths for atom in path)
        if len(forbiddenAtoms) > 0:
            prevTraversed.update(forbiddenAtoms)

        eligiblePaths = []
        for path in contactHash[currentPeriod]:
            lastAtom = path[-1]  # the frontier atom of this trajectory
            atom = molec.GetAtomWithIdx(lastAtom)
            neighbors = atom.GetNeighbors()
            for nbr in neighbors:
                nbrID = nbr.GetIdx()
                if nbrID not in prevTraversed:
                    newPath = path + [nbrID]  # extend trajectory
                    eligiblePaths.append(newPath)
                    prevTraversed.add(nbrID)  # mark as visited within this period expansion

        newPeriod = currentPeriod + 1
        contactHash[newPeriod] = eligiblePaths
        return breadthFirstSearch(molec, period, contactHash, forbiddenAtoms)


In [ ]:
from rdkit.Chem import Draw
CC , molec = initAlkene("CC(C)N(C(=O)O/C=C\[C@H](C)[C@H](O)[C@H]1O[C@@H]2OC(C)(C)O[C@@H]2[C@H]2OC(C)(C)O[C@H]21)C(C)C" , True)
print(CC)
contactHash = breadthFirstSearch(molec, 4 , {0: [[7]]} , [8])
print(contactHash)

In [ ]:
from rdkit import Chem
from rdkit.Chem.Draw import IPythonConsole

# Enable atom indices globally for Jupyter
IPythonConsole.drawOptions.addAtomIndices = True

# Draw a molecule
mol = Chem.MolFromSmiles("CC(C)N(C(=O)O/C=C\[C@H](C)[C@H](O)[C@H]1O[C@@H]2OC(C)(C)O[C@@H]2[C@H]2OC(C)(C)O[C@H]21)C(C)C")

from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display
from PIL import Image
import io

drawer = rdMolDraw2D.MolDraw2DCairo(300, 300)
drawer.drawOptions().addAtomIndices = True
drawer.DrawMolecule(mol)
drawer.FinishDrawing()

img = Image.open(io.BytesIO(drawer.GetDrawingText()))
display(img)

In [ ]:
from rdkit.Chem import rdmolops
path = rdmolops.GetShortestPath(molec, 20, 7)
print(path)
bonds = [mol.GetBondBetweenAtoms(path[i], path[i+1]) for i in range(len(path)-1)]
for bond in bonds:
    bond_type = bond.GetBondType()
    begin_atom = bond.GetBeginAtomIdx()
    end_atom = bond.GetEndAtomIdx()
    print(begin_atom , end_atom)
    print(bond_type)

In [ ]:
idxList = [55,23,12 , 14, 20]
for i in range(len(idxList)-1):
    print(idxList[i] , idxList[i + 1])

In [ ]:
dfMAST = pd.read_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1/alkeneFeaturesMAST.csv")
trans = dfMAST[(dfMAST["HCount"] == 2) & (dfMAST["EvsZ"] == 1)]
cis = dfMAST[(dfMAST["HCount"] == 2) & (dfMAST["EvsZ"] == -1)]
gem = dfMAST[(dfMAST["HCount"] == 2) & (dfMAST["EvsZ"] == 0)]


In [ ]:
trans.to_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1/trans.csv")
cis.to_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1/cis.csv")
gem.to_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1/gem.csv")

In [ ]:
print(np.abs(-1+-3))

In [ ]:
coords1 = np.array([ [3,4,5] , [-3,-4,-5] , [3, -4,5 ] , [-3,-4,5]  , [3,4,-5]])
atoms1 = ["C" , "Si" , "O" , "H" , "N"]



quadLabelsC1 = (coords1[:, 0] > 0).astype(int) + 2 * (coords1[:, 2] > 0).astype(int)
print(quadLabelsC1)
quadrantsC1 = [np.where(quadLabelsC1 == i)[0] for i in range(4)]
print(quadrantsC1)
for quad in quadrantsC1:
    atoms = [atoms1[i] for i in quad]
    print(atoms)

In [ ]:
testHash = {"0" : 0 , "1" : 1}
test = list(testHash.values())
print(test)

In [2]:
import plotly.graph_objects as go
from sklearn.metrics import silhouette_score
def makeFig(xList , yList , zDF , titleStr):
    template = go.layout.Template()
    template.layout.font = dict(family="Arial", size=8, color="black")
    template.layout.plot_bgcolor = "white"
    template.layout.xaxis.linewidth = 5
    template.layout.xaxis.linecolor = "black"
    template.layout.xaxis.showgrid = False
    template.layout.xaxis.tickangle = 45 
    template.layout.yaxis.linewidth = 5
    template.layout.yaxis.linecolor = "black"
    template.layout.yaxis.showgrid = False
    UPENN_SCALE = [
        [0.0, "#990000"],   # red
        [0.5, "#F2F2F2"],   # light neutral midpoint
        [1.0, "#011F5B"]    # blue
    ]
    fig = go.Figure(
        data=go.Heatmap(
            z=zDF,
            x=xList,
            y=yList,
    
            # text overlay
            text=zDF,
            texttemplate="%{text:.4f}",
            textfont=dict(size=10),
    
            colorscale=UPENN_SCALE,
    
            colorbar=dict(
                title=titleStr
            ),
    
            zmid=0
        ),
    
        layout=dict(
            template=template,
    
            # square cells
            yaxis=dict(
                scaleanchor="x",
                scaleratio=1
            ),
    
            # reduce excess whitespace
            margin=dict(
                l=40,
                r=40,
                t=40,
                b=40
            ),
    
            autosize=True
        )
    )
    
    return fig

def calc_silhouette(cluster1, cluster2, metric="euclidean"):


    # Convert to numpy arrays
    cluster1 = np.array(cluster1)
    cluster2 = np.array(cluster2)

    # Combine data
    X = np.vstack([cluster1, cluster2])

    # Create labels
    labels = np.array(
        [0] * len(cluster1) +
        [1] * len(cluster2)
    )

    # Compute silhouette score
    score = silhouette_score(X, labels, metric=metric)

    return score

def silhouette_1d(cluster1, cluster2):

    # Convert to column vectors
    cluster1 = np.array(cluster1).reshape(-1, 1)
    cluster2 = np.array(cluster2).reshape(-1, 1)

    # Combine
    X = np.vstack([cluster1, cluster2])

    # Labels
    labels = np.array(
        [0] * len(cluster1) +
        [1] * len(cluster2)
    )

    return silhouette_score(X, labels)


In [3]:
def insertIntoDataframe(df1 ,  connectingStr ,connectingStr2 ,  df2 , inputStr , indexInterest  ):
    #First create an empty column of zeros at the index of interest and call it inputStr 
    df1.insert(indexInterest, inputStr, 0)
    
    col1 = list(df1[connectingStr])

    col2 = list(df2[connectingStr2])
    insertCol = list(df2[inputStr])
    

    for i , str in enumerate (col1):
        if str in col2:
            index = col2.index(str)
            insertVal = insertCol[index]
            #df1[inputStr][i] = insertVal
            df1.loc[i, inputStr] = insertVal
    return df1

In [15]:
dfMAST = pd.read_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1_altSterics/alkeneFeaturesMAST.csv")


trans = dfMAST[(dfMAST["HCount"] == 2) & (dfMAST["EvsZ"] == 1)]
cis = dfMAST[(dfMAST["HCount"] == 2) & (dfMAST["EvsZ"] == -1)]
gem = dfMAST[(dfMAST["HCount"] == 2) & (dfMAST["EvsZ"] == 0)]
term = dfMAST[(dfMAST["HCount"] == 3) & (dfMAST["EvsZ"] == 0)]
tri = dfMAST[(dfMAST["HCount"] == 1) & (dfMAST["EvsZ"] == 0)]
tet = dfMAST[(dfMAST["HCount"] == 0) & (dfMAST["EvsZ"] == 0)]

print(term.shape)
print(tri.shape)
print(tet.shape)

(567, 116)
(148, 116)
(125, 116)


In [12]:
dfMAST.to_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1_altSterics/alkeneFeaturesMAST.csv")

In [21]:
import re

pattern = re.compile(r"^[^_]+(?:_[^_]+){3}$")

sterCols = [col for col in trans.columns if pattern.match(col) and "Ang" not in col and "slice" not in col]
print(sterCols)

['C_1_1_2.0', 'C_1_1_2.5', 'C_1_1_3.0', 'C_1_1_3.5', 'C_1_1_4.0', 'C_1_2_2.0', 'C_1_2_2.5', 'C_1_2_3.0', 'C_1_2_3.5', 'C_1_2_4.0', 'C_2_1_2.0', 'C_2_1_2.5', 'C_2_1_3.0', 'C_2_1_3.5', 'C_2_1_4.0', 'C_2_2_2.0', 'C_2_2_2.5', 'C_2_2_3.0', 'C_2_2_3.5', 'C_2_2_4.0']


In [ ]:
c1Pattern = re.compile(r"^[^_]+(?:_[^_]+){3}$")

In [22]:

silhouetteHash = {}
for col in sterCols:
    print(col)
    # Extract values
    cisSter = cis[col].dropna().to_numpy()
    transSter = trans[col].dropna().to_numpy()
    gemSter = gem[col].dropna().to_numpy()

    # Reshape for sklearn
    cisArr = cisSter.reshape(-1, 1)
    transArr = transSter.reshape(-1, 1)
    gemArr = gemSter.reshape(-1, 1)

    pairScores = []

    # All pairwise combinations
    combos = [
        (cisArr, transArr),
        (cisArr, gemArr),
        (transArr, gemArr)
    ]

    for c1, c2 in combos:

        X = np.vstack([c1, c2])

        labels = np.array(
            [0] * len(c1) +
            [1] * len(c2)
        )

        score = silhouette_score(X, labels)

        pairScores.append(score)

    # Average silhouette score
    print(pairScores)
    avgScore = np.mean(pairScores)

    silhouetteHash[col] = avgScore
print(silhouetteHash)

C_1_1_2.0
[0.00420260907525772, 0.11243302024341231, 0.13591833797571967]
C_1_1_2.5
[0.008159964129103012, 0.3112460883266418, 0.3630631781538674]
C_1_1_3.0
[0.011513486984503921, 0.3371893205042686, 0.4090113578577179]
C_1_1_3.5
[0.014282380612821848, 0.2513157623056413, 0.3394933768579004]
C_1_1_4.0
[0.017083918177291647, 0.1462220146793459, 0.23749854151150812]
C_1_2_2.0
[0.008382136321047774, 0.4584728927228687, 0.4487976470452778]
C_1_2_2.5
[0.012480202760139744, 0.28116936481669835, 0.3272737856383645]
C_1_2_3.0
[0.015472173873032891, 0.05364231163999697, 0.10756263312270424]
C_1_2_3.5
[0.018369526715454984, 0.00804698934403204, 0.037774794838361075]
C_1_2_4.0
[0.02246983204303638, 0.002280835461440799, 0.027622846525426122]
C_2_1_2.0
[0.009017276241156046, 0.11835808755518756, 0.08796601246334536]
C_2_1_2.5
[0.0409702297090096, 0.15685816186585202, 0.1506421478730847]
C_2_1_3.0
[0.04519035800539494, 0.18948180201344442, 0.17559567260045328]
C_2_1_3.5
[0.029258643311812896, 0.145

In [10]:
fig = makeFig(["2.0Ang" , "2.5Ang",  "3.0Ang"] , ["C1_slice0" , "c1_slice1" , "C1_slice2" , "c1_slice3" , "C2_slice0" , "c2_slice1" , "C2_slice2" , "c2_slice3"] , [[0.013722,
                                                                                                                            0.0820704,0.0697041],
                                                                                                                            [0.6184457,0.4592346,0.2343225],
                                                                                                                            [0.013722,0.050843,0.244512],
                                                                                                                            [0.615106,0.512476,0.4808247],
                                                                                                                            [0.048936,0.05878,0.1968975],
                                                                                                                            [0.017983,0.03807,0.08727],
                                                                                                                            [0.625632,0.48394,0.3867529],
                                                                                                                            [0.017983,0.035005,0.2188126]]
                                                                                                                            , "Sliced Oranges")
fig.show()

In [11]:


fig = makeFig(["2.5Ang" , "3.0Ang",  "3.5Ang" , "4.0Ang" , "4.5Ang"] , ["minCap" , "maxCap" , "minSemiPi" , "maxSemiPi"] , [[0.5121620111725528,
                                                                                                                            0.5431787638248692,
                                                                                                                            0.5016164492078204,
                                                                                                                            0.46031534992733875,
                                                                                                                            0.49235012175745974],
                                                                                                                            [0.37627977335709484,
                                                                                                                            0.34165713579929363,
                                                                                                                            0.25746833491140264,
                                                                                                                            0.21022264475102895,
                                                                                                                            0.16692672275335543,
                                                                                                                            ],
                                                                                                                            [0.5942882972474616,
                                                                                                                            0.5983457885788699,
                                                                                                                            0.5739727844954942,
                                                                                                                            0.5264384788339423,
                                                                                                                            0.49235012175745974],
                                                                                                                            [0.278635034657376,
                                                                                                                            0.1774894238206641,
                                                                                                                            0.19733368627918804,
                                                                                                                            0.24268783096397395,
                                                                                                                            0.2167966441486169]]
                                                                                                                            , "Capped Semi Cylinders")
fig.show()

In [ ]:


fig = makeFig(["2.0Ang" , "3.5Ang",  "4.0Ang" , "4.5Ang" , "5.0Ang"] , ["L" , "B1" , "B5"] , [[0.010066354365419363 , 
                                                                                    0.00912460103840185, 
                                                                                    0.08709543410158192 , 
                                                                                    0.07757634533424826 , 
                                                                                   0.063342482654341083] , 
                                                                                   [0.39741272685977097,
                                                                                    0.39140179087908751
                                                                                    ,0.39161951851830185
                                                                                    ,0.39949917500642856
                                                                                   ,0.399286704327235754] ,
                                                                                   [0.09134734921542657,
                                                                                    0.12140148759475,
                                                                                    0.08154401751215312,
                                                                                    0.07077687331503878,
                                                                                   0.067917458179072390]], "Sterimol")
fig.show()

In [ ]:
fig = makeFig(["2.0Ang" , "2.5Ang" , "3.0Ang" , "3.5Ang","4.0Ang" ] , ["%Vbur_C1" , "%Vbur_C2"  , "delta%Vbur" , "mean%Vbur"] , [[0.4007754699790427 ,
                                                                                                                                 0.3484014322340454 , 
                                                                                                                                 0.26745658557938773,
                                                                                                                                 0.1986966967466801,
                                                                                                                                 0.13247637966457676] , 
                                                                                                                                [0.35177259520166543 ,
                                                                                                                                0.3551353938878153 , 
                                                                                                                                0.2871640366093739 , 
                                                                                                                                0.17436488546042353,
                                                                                                                                0.06046172578205387
                                                                                                                                ] , 
                                                                                                                                [0.26915741356573736,
                                                                                                                                0.07345798738403288,
                                                                                                                                -0.0053140286969522625,
                                                                                                                                0.02170186610255037,
                                                                                                                                0.07370344390377508] , 
                                                                                                                                [0.3707432990327277,
                                                                                                                                0.34151794535545904,
                                                                                                                                0.2768159752399398,
                                                                                                                                0.21022080126132595,
                                                                                                                                0.12025600992804213]], "Spherical Buried Volume")
fig.show()

In [ ]:
fig = makeFig(["Period" , "Period2" , "Period3" , "period4"] , ["Topology_C1" , "Topology_C2"  , "meanTopology" , "deltaTopology"] , [[0.5837020998966921,
                                                                                                                     0.5676787134265829,
                                                                                                                     0.5625897336256841,
                                                                                                                     0.5605301631553873] , 
                                                                                                                     [0.621990294930773,
                                                                                                                     0.6136232471311199 , 
                                                                                                                     0.6109540722213995 , 
                                                                                                                     0.6107799712154748] , 
                                                                                                                    [0.10641497502918733,
                                                                                                                    0.0813925409267451,
                                                                                                                    0.07227152972751717,
                                                                                                                    0.06846851754939305],
                                                                                                                     [0.6137780391466997,
                                                                                                                     0.6137780391466997,
                                                                                                                     0.6124323405707338,
                                                                                                                     0.612072476794167] ], "TSEI")
fig.show()

In [ ]:
dir1 = "/media/danny/KINGSTON/Stahl/chemistriesDatasets/DRC_DJW_SSS_P1_v1/alkeneCategories/"
transDF = pd.read_csv(dir1 + "transAlkenes.csv")
cisDF = pd.read_csv(dir1 + "cisAlkenes.csv")
gemDF = pd.read_csv(dir1 + "gemAlkenes.csv")

transTSEI = np.column_stack((np.array(transDF["deltaTopology_2"]),np.array(transDF["minTopology_1"])))
cisTSEI = np.column_stack((np.array(cisDF["deltaTopology_2"]),np.array(cisDF["minTopology_1"])))
gemTSEI = np.column_stack((np.array(gemDF["deltaTopology_2"]),np.array(gemDF["minTopology_1"])))
TvC = calc_silhouette(transTSEI, cisTSEI, metric="euclidean")
TvG = calc_silhouette(transTSEI, gemTSEI, metric="euclidean")
CvG = calc_silhouette(cisTSEI, gemTSEI, metric="euclidean")
print(TvC, TvG , CvG)

In [ ]:
transSter = np.column_stack((np.array(transDF["sterB1_4.5"]),np.array(transDF["sterB5_3.0_lowE"])))
cisSter = np.column_stack((np.array(cisDF["sterB1_4.5"]),np.array(cisDF["sterB5_3.0_lowE"])))
gemSter = np.column_stack((np.array(gemDF["sterB1_4.5"]),np.array(gemDF["sterB5_3.0_lowE"])))
TvC = calc_silhouette(transSter, cisSter, metric="euclidean")
TvG = calc_silhouette(transSter, gemSter, metric="euclidean")
CvG = calc_silhouette(cisSter, gemSter, metric="euclidean")
print(TvC, TvG , CvG)
transVbur = np.column_stack((np.array(transDF["2.0_Ang_Vburr_Cmn"]),np.array(transDF["2.0_Ang_Vburr_mean"])))
cisVbur = np.column_stack((np.array(cisDF["2.0_Ang_Vburr_Cmn"]),np.array(cisDF["2.0_Ang_Vburr_mean"])))
gemVbur = np.column_stack((np.array(gemDF["2.0_Ang_Vburr_Cmn"]),np.array(gemDF["2.0_Ang_Vburr_mean"])))
TvC = calc_silhouette(transVbur, cisVbur, metric="euclidean")
TvG = calc_silhouette(transVbur, gemVbur, metric="euclidean")
CvG = calc_silhouette(cisVbur, gemVbur, metric="euclidean")
print(TvC, TvG , CvG)
transSemi = np.column_stack((np.array(transDF["Vbur_MinSemi_Pi_3.0"]),np.array(transDF["Vbur_MinCap_3.0"])))
cisSemi = np.column_stack((np.array(cisDF["Vbur_MinSemi_Pi_3.0"]),np.array(cisDF["Vbur_MinCap_3.0"])))
gemSemi = np.column_stack((np.array(gemDF["Vbur_MinSemi_Pi_3.0"]),np.array(gemDF["Vbur_MinCap_3.0"])))
TvC = calc_silhouette(transSemi, cisSemi, metric="euclidean")
TvG = calc_silhouette(transSemi, gemSemi, metric="euclidean")
CvG = calc_silhouette(cisSemi, gemSemi, metric="euclidean")
print(TvC, TvG , CvG)

In [ ]:
dfMAST = pd.read_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1_v02/alkeneFeaturesMAST.csv")
deltaSemiPi = list(abs(np.array(dfMAST["Vbur_MinSemi_Pi_2.5"]) - np.array(dfMAST["Vbur_MaxSemi_Pi_2.5"])))
dfMAST["deltaSemiPi_2.5"] = deltaSemiPi
deltaCap = list(abs(np.array(dfMAST["Vbur_MaxCap_3.0"]) - np.array(dfMAST["Vbur_MinCap_3.0"])))
dfMAST["deltaCap_3.0"] = deltaCap
transDF = dfMAST[(dfMAST["HCount"] == 2) & (dfMAST["EvsZ"] == 1)]
cisDF = dfMAST[(dfMAST["HCount"] == 2) & (dfMAST["EvsZ"] == -1)]
gemDF = dfMAST[(dfMAST["HCount"] == 2) & (dfMAST["EvsZ"] == 0)]
'''
transSemi = np.column_stack((np.array(transDF["Vbur_MinSemi_Pi_2.5"]),np.array(transDF["Vbur_MinCap_3.0"])))
cisSemi = np.column_stack((np.array(cisDF["Vbur_MinSemi_Pi_2.5"]),np.array(cisDF["Vbur_MinCap_3.0"])))
gemSemi = np.column_stack((np.array(gemDF["Vbur_MinSemi_Pi_2.5"]),np.array(gemDF["Vbur_MinCap_3.0"])))
TvC = calc_silhouette(transSemi, cisSemi, metric="euclidean")
TvG = calc_silhouette(transSemi, gemSemi, metric="euclidean")
CvG = calc_silhouette(cisSemi, gemSemi, metric="euclidean")
print(TvC, TvG , CvG)
'''

In [ ]:
transDF.to_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1_v02/deltaSemidiffCategories/trans.csv")
cisDF.to_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1_v02/deltaSemidiffCategories/cis.csv")
gemDF.to_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1_v02/deltaSemidiffCategories/gem.csv")

In [ ]:
term = dfMAST[(dfMAST["HCount"] == 3) & (dfMAST["EvsZ"] == 0)]
triE = dfMAST[(dfMAST["HCount"] == 1) & (dfMAST["EvsZ"] == 1)]
tetE = dfMAST[(dfMAST["HCount"] == 0) & (dfMAST["EvsZ"] == 1)]
triZ = dfMAST[(dfMAST["HCount"] == 1) & (dfMAST["EvsZ"] == -1)]
tetZ = dfMAST[(dfMAST["HCount"] == 0) & (dfMAST["EvsZ"] == -1)]
tet0 = dfMAST[(dfMAST["HCount"] == 0) & (dfMAST["EvsZ"] == 0)]
tri0 = dfMAST[(dfMAST["HCount"] == 1) & (dfMAST["EvsZ"] == 0)]
term.to_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1_v02/deltaSemidiffCategories/term.csv")
triE.to_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1_v02/deltaSemidiffCategories/triE.csv")
tetE.to_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1_v02/deltaSemidiffCategories/tetE.csv")
triZ.to_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1_v02/deltaSemidiffCategories/triZ.csv")
tetZ.to_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1_v02/deltaSemidiffCategories/tetZ.csv")
tri0.to_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1_v02/deltaSemidiffCategories/tri0.csv")
tet0.to_csv("/home/danny/Downloads/alkeneSpaceMap_MAST/fastCalculations/alkeneLogs/DRC_DJW_SSS_P1_v02/deltaSemidiffCategories/tet0.csv")